# Clone via ArcGIS Pro — keep style + group structure

Run inside **ArcGIS Pro** on map `to_clone`.

1. Export data from each **source** group (e.g. `GAR`) → `{Group}.gdb` (normalized FC names)
2. Duplicate the group with `SaveToLayerFile` (preserves symbology + nesting)
3. Rename group to `GAR_stg` / `GAR_dev` and retarget layers to the local GDB

Layer **display names** stay the same; only the group name gets the suffix.

In [ ]:
# Cell 1 — project + map
import os
import re
import tempfile
from collections import defaultdict

import arcpy

MAP_NAME = "to_clone"
OVERWRITE = True
SUFFIXES = ["_stg", "_dev"]

aprx = arcpy.mp.ArcGISProject("CURRENT")
arcpy.env.overwriteOutput = OVERWRITE

maps = [m for m in aprx.listMaps() if m.name == MAP_NAME]
if len(maps) != 1:
    raise ValueError(f"Need exactly 1 map named {MAP_NAME!r}, found {len(maps)}")
clone_map = maps[0]

out_dir = aprx.homeFolder or os.path.dirname(aprx.defaultGeodatabase)
print("project:", aprx.filePath)
print("map:", clone_map.name)
print("out_dir:", out_dir)
print("suffixes:", SUFFIXES)

In [ ]:
# Cell 2 — source groups only (skip GAR_stg / local GDB leftovers)

def normalize_name(name: str, *, max_len: int = 64) -> str:
    cleaned = re.sub(r"[^A-Za-z0-9_]+", "_", (name or "").strip()).strip("_")
    if not cleaned:
        cleaned = "layer"
    if cleaned[0].isdigit():
        cleaned = f"f_{cleaned}"
    return cleaned[:max_len]


def unique_name(base: str, used: set) -> str:
    name = base
    i = 2
    while name.lower() in used:
        suffix = f"_{i}"
        name = f"{base[: 64 - len(suffix)]}{suffix}"
        i += 1
    used.add(name.lower())
    return name


def is_suffixed_group(name: str) -> bool:
    return any((name or "").endswith(s) for s in SUFFIXES)


def is_file_gdb_layer(lyr) -> bool:
    try:
        wf = (lyr.connectionProperties or {}).get("workspace_factory") or ""
        return "file geodatabase" in str(wf).lower() or str(wf).upper() == "FILEGDB"
    except Exception:
        ds = (getattr(lyr, "dataSource", None) or "").lower()
        return ds.endswith(".gdb") or "\\" in ds and ".gdb\\" in ds.replace("/", "\\").lower()


def top_source_groups(map_obj):
    """Top-level groups that are not already GAR_stg / GAR_dev."""
    groups = []
    for lyr in map_obj.listLayers():
        if not lyr.isGroupLayer:
            continue
        if "\\" in lyr.longName:
            continue  # nested group, not top-level
        if is_suffixed_group(lyr.name):
            continue
        groups.append(lyr)
    return groups


def feature_layers_in_group(group_lyr):
    """Leaf feature layers under a group, skipping local FileGDB leftovers."""
    rows = []
    seen_ds = set()
    for lyr in group_lyr.listLayers():
        if lyr.isGroupLayer:
            continue
        if not lyr.supports("dataSource"):
            continue
        if is_file_gdb_layer(lyr):
            continue
        ds = lyr.dataSource
        if ds in seen_ds:
            continue
        seen_ds.add(ds)
        rows.append(lyr)
    return rows


source_groups = top_source_groups(clone_map)
if not source_groups:
    raise ValueError(f"No source groups found in map {MAP_NAME!r}")

export_plan = {}  # gdb_name -> {group_lyr, group_raw, layers:[{lyr, layer_name, fc_name, data_source}]}

for group_lyr in source_groups:
    group_raw = group_lyr.name
    gdb_name = normalize_name(group_raw)
    used = set()
    layers = []
    for lyr in feature_layers_in_group(group_lyr):
        fc_name = unique_name(normalize_name(lyr.name), used)
        layers.append(
            {
                "lyr": lyr,
                "layer_name": lyr.name,
                "fc_name": fc_name,
                "data_source": lyr.dataSource,
                "long_name": lyr.longName,
            }
        )
    if not layers:
        print("skip empty group:", group_raw)
        continue
    export_plan[gdb_name] = {
        "group_lyr": group_lyr,
        "group_raw": group_raw,
        "layers": layers,
    }
    print(f"{gdb_name}.gdb  (group={group_raw!r}, layers={len(layers)})")
    for r in layers:
        print(f"  {r['layer_name']!r} -> {r['fc_name']}")

if not export_plan:
    raise ValueError("No exportable layers (all skipped as FileGDB or empty)")

In [ ]:
# Cell 3 — create File GDBs + ExportFeatures (data only)
exported = {}

for gdb_name, info in export_plan.items():
    gdb_path = os.path.join(out_dir, f"{gdb_name}.gdb")
    if arcpy.Exists(gdb_path):
        if OVERWRITE:
            arcpy.management.Delete(gdb_path)
        else:
            raise RuntimeError(f"GDB already exists: {gdb_path}")
    arcpy.management.CreateFileGDB(out_dir, f"{gdb_name}.gdb")
    print("created", gdb_path)

    out_layers = []
    for r in info["layers"]:
        out_fc = os.path.join(gdb_path, r["fc_name"])
        print(f"  export {r['long_name']!r} -> {r['fc_name']}")
        arcpy.conversion.ExportFeatures(
            in_features=r["lyr"],
            out_features=out_fc,
            use_field_alias_as_name="NOT_USE_ALIAS",
        )
        count = int(arcpy.management.GetCount(out_fc)[0])
        out_layers.append({**r, "count": count})
        print(f"    count={count}")

    exported[gdb_name] = {
        "gdb_path": gdb_path,
        "group_raw": info["group_raw"],
        "group_lyr": info["group_lyr"],
        "layers": out_layers,
    }

exported.keys()

In [ ]:
# Cell 4 — duplicate group via .lyrx (keeps style + nesting), retarget to GDB

def remove_top_group(map_obj, group_name: str) -> bool:
    for lyr in list(map_obj.listLayers()):
        if lyr.isGroupLayer and lyr.name == group_name and "\\" not in lyr.longName:
            map_obj.removeLayer(lyr)
            print("removed existing group:", group_name)
            return True
    return False


def retarget_group_to_gdb(group_lyr, gdb_path: str, layers_meta: list) -> int:
    """Point each leaf layer at its exported FC; match by original dataSource then name."""
    by_ds = {r["data_source"]: r["fc_name"] for r in layers_meta}
    by_name = defaultdict(list)
    for r in layers_meta:
        by_name[r["layer_name"]].append(r["fc_name"])

    updated = 0
    for lyr in group_lyr.listLayers():
        if lyr.isGroupLayer or not lyr.supports("dataSource"):
            continue

        fc_name = by_ds.get(lyr.dataSource)
        if not fc_name:
            names = by_name.get(lyr.name) or []
            fc_name = names.pop(0) if names else None
        if not fc_name:
            print("  ! no FC match for", lyr.longName)
            continue

        fc_path = os.path.join(gdb_path, fc_name)
        if not arcpy.Exists(fc_path):
            raise FileNotFoundError(fc_path)

        # Prefer replaceDataSource (simple); fall back to updateConnectionProperties
        try:
            lyr.replaceDataSource(gdb_path, "FILEGDB_WORKSPACE", fc_name, False)
        except Exception:
            new_cp = {
                "workspace_factory": "File Geodatabase",
                "connection_info": {"database": gdb_path},
                "dataset": fc_name,
            }
            lyr.updateConnectionProperties(lyr.connectionProperties, new_cp)

        # Keep original display name (lyrx already has it; enforce after retarget)
        # Find original name from meta
        for r in layers_meta:
            if r["fc_name"] == fc_name:
                lyr.name = r["layer_name"]
                break
        updated += 1
        print(f"  retarget {lyr.name!r} -> {fc_name}")
    return updated


added_groups = {}

for gdb_name, info in exported.items():
    group_raw = info["group_raw"]
    source_group = info["group_lyr"]
    gdb_path = info["gdb_path"]

    # Refresh group object from map (stale refs after long export)
    refreshed = [
        lyr
        for lyr in clone_map.listLayers()
        if lyr.isGroupLayer and lyr.name == group_raw and "\\" not in lyr.longName
    ]
    if not refreshed:
        raise RuntimeError(f"Source group {group_raw!r} not found on map")
    source_group = refreshed[0]

    for suffix in SUFFIXES:
        group_name = f"{group_raw}{suffix}"

        if OVERWRITE:
            remove_top_group(clone_map, group_name)
        elif any(
            lyr.isGroupLayer and lyr.name == group_name and "\\" not in lyr.longName
            for lyr in clone_map.listLayers()
        ):
            print("skip existing group:", group_name)
            continue

        lyrx = os.path.join(tempfile.gettempdir(), f"{normalize_name(group_name)}.lyrx")
        if arcpy.Exists(lyrx):
            arcpy.management.Delete(lyrx)

        arcpy.management.SaveToLayerFile(source_group, lyrx, "RELATIVE")
        added = clone_map.addDataFromPath(lyrx)
        # addDataFromPath may return the group or a list-like; normalize
        if isinstance(added, (list, tuple)):
            added = added[0]
        added.name = group_name
        print("created group (styled copy):", group_name)

        n = retarget_group_to_gdb(added, gdb_path, info["layers"])
        added_groups[group_name] = {"gdb_path": gdb_path, "retargeted": n}

        try:
            arcpy.management.Delete(lyrx)
        except Exception:
            pass

aprx.save()
print("saved project")
added_groups